In [251]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

In [252]:
bundle_version = '26.2.1.5'

Definir los archivos CSV que se usaran para los escenarios online y curve

Para online: SettleAnalysis


Para curve: EngageAnalysis

In [253]:
#vehicle settle para Online runs 

#Engaged Analysis para Curve runs

online_csv_path = '/Users/josedejesuspena/Documents/files to test with AG report/26.2.1.5/Auto-guidance - Summary/Results Metrics - EngagedAnalysis.csv'
curve_csv_path  = '/Users/josedejesuspena/Documents/files to test with AG report/26.2.1.5/Auto-guidance - Summary/Results Metrics - EngagedAnalysis.csv'

In [254]:
# Se especifica aqui, Software version y el nombre del archivo excel que se exportara

filename_excel_report = bundle_version + '_Test Status Report.xlsx' 


#Se leen los archivos que tienen la informacion para el analisis 


df_OnlineAnalysis_csv = pd.read_csv(online_csv_path)  
df_EngagedAnalysis_csv = pd.read_csv(curve_csv_path)



#Specs for peaks and absolutes
spec_peaks = .05 # Spec for peaks - 5 cm  
spec_abs = .025 # Spec for abs - 2.5 cm  
spec_std = .0295 # Spec for standard deviation - 2.95cm  


In [255]:
df_OnlineAnalysis_csv.columns

Index(['Kringle Event UUID', 'Kringle Event Start Datetime (UTC)',
       'AVGVelocity (mps)', 'ABSEngageAngle (deg)', 'ABSEngageOffset (m)',
       'Data Source', 'Test Case Classification',
       'Acceptance Criteria Passed (All)', 'IMPSettlingDistance (m)',
       'IMPSettlingTime (s)', 'ABSIMPEngageOffset (m)',
       'AVGABSCommandAngle (deg)', 'AVGABSIMPXTE (m)', 'AVGABSRoll (deg)',
       'AVGABSSteeringAngle (deg)', 'AVGABSXTE (m)', 'AVGCommandAngle (deg)',
       'AVGIMPXTE (m)', 'AVGRoll (deg)', 'AVGSpeed (mps)', 'AVGSpeedkph (kph)',
       'AVGSteeringAngle (deg)', 'AVGVelocitykph (kph)', 'AVGXTE (m)',
       'Distance (m)', 'Duration (s)', 'IMPOvershoot (m)',
       'MAXABSCommandAngle (deg)', 'MAXABSCommandSlew (dps)',
       'MAXABSIMPXTE (m)', 'MAXABSRoll (deg)', 'MAXABSSteeringAngle (deg)',
       'MAXABSSteeringSlew (dps)', 'MAXABSXTE (m)', 'MAXIMPXTE (m)',
       'MAXXTE (m)', 'MINABSIMPXTE (m)', 'MINABSRoll (deg)', 'MINABSXTE (m)',
       'MINIMPXTE (m)', 'MINXTE (m

In [256]:
# Lista con el nombre de las columnas que nos seran utiles para el analisis. 

# Estas columnas se usaran para crear el df_online y el df_curve
perf_metrics = ['Data Source','MAXABSIMPXTE (m)', 'AVGABSIMPXTE (m)','STDIMPXTE (m)']

In [257]:
df_OnlineAnalysis_csv['Data Source']

0                Run 1 - straight_run1_1.25mph_2
1                Run 2 - straight_run1_1.25mph_3
2                 Run 3 - straight_run2_3.1mph_1
3                 Run 4 - straight_run2_3.1mph_2
4                 Run 5 - straight_run2_3.1mph_3
5                 Run 6 - straight_run3_6.2mph_1
6                 Run 7 - straight_run3_6.2mph_2
7                 Run 8 - straight_run3_6.2mph_3
8                 Run 9 - straight_run4_9.3mph_1
9                Run 10 - straight_run4_9.3mph_2
10               Run 11 - straight_run4_9.3mph_3
11              Run 12 - straight_run5_12.5mph_1
12              Run 13 - straight_run5_12.5mph_2
13              Run 14 - straight_run5_12.5mph_3
14                Run 15 - curve1deg_run6_5mph_1
15                Run 16 - curve1deg_run6_5mph_2
16                Run 17 - curve1deg_run6_5mph_3
17                Run 18 - curve2deg_run7_5mph_1
18                Run 19 - curve2deg_run7_5mph_2
19                Run 20 - curve2deg_run7_5mph_3
20                Ru

Se crea el dataframe que contendra las corridas para el df_online

In [258]:
# Se crea una lista donde se guardan los rows que devuelven true cuando se lee la palabra straight
online_runs = df_OnlineAnalysis_csv["Data Source"].str.contains('straight')


df_online = df_OnlineAnalysis_csv.loc[online_runs][perf_metrics]
df_online.index = range(1,len(df_online) + 1)
df_online.rename(columns={'Data Source': 'Run Description'},inplace=True)

Verifica que solo se vean las corridas straight en el df_online

In [259]:
df_online

,Run Description,MAXABSIMPXTE (m),AVGABSIMPXTE (m),STDIMPXTE (m)
1,Run 1 - straight_run1_1.25mph_2,0.035,0.013,0.015
2,Run 2 - straight_run1_1.25mph_3,0.034,0.012,0.014
3,Run 3 - straight_run2_3.1mph_1,0.049,0.016,0.020
4,Run 4 - straight_run2_3.1mph_2,0.040,0.016,0.018
5,Run 5 - straight_run2_3.1mph_3,0.032,0.013,0.015
6,Run 6 - straight_run3_6.2mph_1,0.134,0.049,0.059
7,Run 7 - straight_run3_6.2mph_2,0.094,0.043,0.052
8,Run 8 - straight_run3_6.2mph_3,0.098,0.038,0.047
9,Run 9 - straight_run4_9.3mph_1,0.154,0.057,0.070
10,Run 10 - straight_run4_9.3mph_2,0.192,0.071,0.084


Se crea el dataframe que tendra las corridas para el df_curve

In [260]:
df_EngagedAnalysis_csv['Data Source']

0                Run 1 - straight_run1_1.25mph_2
1                Run 2 - straight_run1_1.25mph_3
2                 Run 3 - straight_run2_3.1mph_1
3                 Run 4 - straight_run2_3.1mph_2
4                 Run 5 - straight_run2_3.1mph_3
5                 Run 6 - straight_run3_6.2mph_1
6                 Run 7 - straight_run3_6.2mph_2
7                 Run 8 - straight_run3_6.2mph_3
8                 Run 9 - straight_run4_9.3mph_1
9                Run 10 - straight_run4_9.3mph_2
10               Run 11 - straight_run4_9.3mph_3
11              Run 12 - straight_run5_12.5mph_1
12              Run 13 - straight_run5_12.5mph_2
13              Run 14 - straight_run5_12.5mph_3
14                Run 15 - curve1deg_run6_5mph_1
15                Run 16 - curve1deg_run6_5mph_2
16                Run 17 - curve1deg_run6_5mph_3
17                Run 18 - curve2deg_run7_5mph_1
18                Run 19 - curve2deg_run7_5mph_2
19                Run 20 - curve2deg_run7_5mph_3
20                Ru

In [261]:
# Se crea una lista donde se guardan los rows que devuelven true cuando se lee la palabra straight
curve_runs = df_EngagedAnalysis_csv["Data Source"].str.contains('curve')


df_curve = df_EngagedAnalysis_csv.loc[curve_runs][perf_metrics]
df_curve.index = range(1,len(df_curve) + 1)
df_curve.rename(columns={'Data Source': 'Run Description'},inplace=True)

Verifica que solo se vean las corridas curve en el df_curve

In [262]:
print(df_curve)

                  Run Description  MAXABSIMPXTE (m)  AVGABSIMPXTE (m)  \
1  Run 15 - curve1deg_run6_5mph_1             0.099             0.034   
2  Run 16 - curve1deg_run6_5mph_2             0.100             0.036   
3  Run 17 - curve1deg_run6_5mph_3             0.142             0.037   
4  Run 18 - curve2deg_run7_5mph_1             0.155             0.067   
5  Run 19 - curve2deg_run7_5mph_2             0.260             0.073   
6  Run 20 - curve2deg_run7_5mph_3             0.273             0.069   
7  Run 21 - curve1deg_run8_8mph_1             0.152             0.054   
8  Run 22 - curve1deg_run8_8mph_2             0.139             0.048   
9  Run 23 - curve1deg_run8_8mph_3             0.167             0.053   

   STDIMPXTE (m)  
1          0.040  
2          0.043  
3          0.048  
4          0.074  
5          0.096  
6          0.095  
7          0.067  
8          0.057  
9          0.062  


Se crean los dataframes

df_perf_type : Contiene el tipo de online perfomance a evaluar, en este caso Straight

df_speeds_online : Contiene las velocidades de cada corrida

df_runs_number_online : Contine el numero de runs. Para online Straights seran 5.

df_peaks_avg_online : Contiene el average MAX por cada conjunto de muestras.

df_abs_avg_online: Contiene el average de XTE por cada conjunto de muestras
 



In [263]:
# Se crea el df que contendra los promedios de las corridas ONLINE unicamente para peaks o MAXABS

df_perf_type = pd.DataFrame(index= range(0,5), columns= ["Online Performance"]) 
df_speeds_online = pd.DataFrame(index= range(0,5), columns= ["Speed"]) 
df_runs_number_online = pd.DataFrame(index= range(0,5), columns= ["Run #"]) 
df_peaks_avg_online =  pd.DataFrame(index= range(0,5), columns= ["Max Average XTE  of runs (m)"]) 
df_abs_avg_online =  pd.DataFrame(index= range(0,5), columns= ["Absolute Average XTE of runs (m)"]) 
df_std_avg_online =  pd.DataFrame(index= range(0,5), columns= ["Standard Deviation Average of runs (m)"]) 



In [264]:
df_online

,Run Description,MAXABSIMPXTE (m),AVGABSIMPXTE (m),STDIMPXTE (m)
1,Run 1 - straight_run1_1.25mph_2,0.035,0.013,0.015
2,Run 2 - straight_run1_1.25mph_3,0.034,0.012,0.014
3,Run 3 - straight_run2_3.1mph_1,0.049,0.016,0.020
4,Run 4 - straight_run2_3.1mph_2,0.040,0.016,0.018
5,Run 5 - straight_run2_3.1mph_3,0.032,0.013,0.015
6,Run 6 - straight_run3_6.2mph_1,0.134,0.049,0.059
7,Run 7 - straight_run3_6.2mph_2,0.094,0.043,0.052
8,Run 8 - straight_run3_6.2mph_3,0.098,0.038,0.047
9,Run 9 - straight_run4_9.3mph_1,0.154,0.057,0.070
10,Run 10 - straight_run4_9.3mph_2,0.192,0.071,0.084


In [265]:
#scenarios : Es una lista que servira para separar las corridas en bloques. 
#            El nombre de los archivos deben contener este texto para separar.

scenarios = ["1.25mph", "3.1mph", "6.2mph", "9.3mph", "12.5mph"] 

#runs_online: Es una lista que servira para llenar el df_runs_number_online con los numeros del 1 al 5.
runs_online = range(1,6)


#avg_max es un diccionario, que llevara por key cada valor de esceanrio y por clave, el promedio
        # de las corridas para ese escenario.
avg_max = {
    s: round(
        df_online.loc[
            df_online["Run Description"].str.contains(s, na=False),
            "MAXABSIMPXTE (m)"
        ].mean(),
        3
    )
    for s in scenarios
}
avg_max = list(avg_max.values())


# Se hace el mismo proceso para avg_abs. 
avg_abs = {
    a: round(
        df_online.loc[
            df_online["Run Description"].str.contains(a, na=False),
            "AVGABSIMPXTE (m)"
        ].mean(),
        3
    )
    for a in scenarios
}


avg_abs_results = list(avg_abs.values())



# Se hace el mismo proceso para stadard deviation. 
avg_std = {
    a: round(
        df_online.loc[
            df_online["Run Description"].str.contains(a, na=False),
            "STDIMPXTE (m)"
        ].mean(),
        3
    )
    for a in scenarios
}


avg_std_results = list(avg_std.values())

In [266]:
#Aqui asignaremos en cada uno de los renglones la informacion que necesitamos para cada df.

for i in range(5):
    df_perf_type.loc[i,'Online Performance'] = 'Straight'
    df_speeds_online.loc[i,'Speed'] = scenarios[i]
    df_runs_number_online.loc[i,'Run #'] = runs_online[i]
    df_peaks_avg_online.loc[i,'Max Average XTE  of runs (m)'] = avg_max[i]
    df_abs_avg_online.loc[i, 'Absolute Average XTE of runs (m)'] = avg_abs_results[i]
    df_std_avg_online.loc[i, 'Standard Deviation Average of runs (m)'] = avg_std_results[i]


In [267]:
#Concatemos de manera horizonal cada uno de los dataframes para generar df_output_online

df_output_online = pd.concat([
                                df_perf_type,
                                df_speeds_online,
                                df_runs_number_online,
                                df_peaks_avg_online,
                                df_abs_avg_online,
                                df_std_avg_online], axis=1)

In [268]:
df_output_online

,Online Performance,Speed,Run #,Max Average XTE of runs (m),Absolute Average XTE of runs (m),Standard Deviation Average of runs (m)
0,Straight,1.25mph,1,0.034,0.012,0.014
1,Straight,3.1mph,2,0.04,0.015,0.018
2,Straight,6.2mph,3,0.109,0.043,0.053
3,Straight,9.3mph,4,0.205,0.075,0.089
4,Straight,12.5mph,5,0.218,0.088,0.102


Realizamos ahora la tabla para los scenarios de curvas. 
Los dataframes utilizados tienen el mismo objetivo que los creados para straight runs, solo llevan como distintivo el _curve

In [269]:
# Se crea el df que contendra los promedios de las corridas Curve unicamente para peaks o MAXABS

df_perf_type = pd.DataFrame(index= range(0,3), columns= ["Online Performance"]) 
df_speeds_curvatures = pd.DataFrame(index= range(0,3), columns= ["Speed / Curvature"]) 
df_runs_number_curve = pd.DataFrame(index= range(0,3), columns= ["Run #"]) 
df_peaks_avg_curve =  pd.DataFrame(index= range(0,3), columns= ["Max Average XTE  of runs (m)"]) 
df_abs_avg_curve =  pd.DataFrame(index= range(0,3), columns= ["Absolute Average XTE of runs (m)"]) 
df_std_avg_curve =  pd.DataFrame(index= range(0,3), columns= ["Standard Deviation Average of runs (m)"]) 

In [270]:
df_abs_avg_curve

,Absolute Average XTE of runs (m)
0,NaN
1,NaN
2,NaN


In [271]:
df_std_avg_curve

,Standard Deviation Average of runs (m)
0,NaN
1,NaN
2,NaN


In [272]:
df_curve['Run Description']

1    Run 15 - curve1deg_run6_5mph_1
2    Run 16 - curve1deg_run6_5mph_2
3    Run 17 - curve1deg_run6_5mph_3
4    Run 18 - curve2deg_run7_5mph_1
5    Run 19 - curve2deg_run7_5mph_2
6    Run 20 - curve2deg_run7_5mph_3
7    Run 21 - curve1deg_run8_8mph_1
8    Run 22 - curve1deg_run8_8mph_2
9    Run 23 - curve1deg_run8_8mph_3
Name: Run Description, dtype: str

In [273]:
df_curve.columns

Index(['Run Description', 'MAXABSIMPXTE (m)', 'AVGABSIMPXTE (m)',
       'STDIMPXTE (m)'],
      dtype='str')

In [274]:
# scenarios: Es la lista que contendra las strings para separar las corridas en bloques y 
            #con eso calcular el promedio

scenarios = ["run6_5mph", 
             "run7_5mph", 
             "run8_8mph"]

# speeds_curvatures: Esta lista solo contiene las velocidades y las curvas para 
#  llenar el df_speeds_curvatures

speeds_curvatures = ["5mph / 1 deg/m", "5 mph / 2deg/m", "8mph / 1 deg/m"]

# runs_curve: Es la lista que serivira para llenar el df_runs_number. Solo es la sucesion de 1 a 4.
runs_curve = range(1,4)

avg_max = {
    s: round(
        df_curve.loc[
            df_curve["Run Description"].str.contains(s, na=False),
            "MAXABSIMPXTE (m)"
        ].mean(),
        3
    )
    for s in scenarios
}
avg_max = list(avg_max.values())

avg_abs = {
    a: round(
        df_curve.loc[
            df_curve["Run Description"].str.contains(a, na=False),
            "AVGABSIMPXTE (m)"
        ].mean(),
        3
    )
    for a in scenarios
}


avg_abs_results = list(avg_abs.values())

# Se hace el mismo proceso para stadard deviation. 
avg_std = {
    a: round(
        df_curve.loc[
            df_curve["Run Description"].str.contains(a, na=False),
            "STDIMPXTE (m)"
        ].mean(),
        3
    )
    for a in scenarios
}


avg_std_results = list(avg_std.values())

In [275]:
avg_std_results

[np.float64(0.044), np.float64(0.088), np.float64(0.062)]

In [276]:
# Se colocan los resultados obtenidos en cada uno de los dataframes:
# Para df_perf_type solo se llenar cada row con el string Curve.
# Para df_speeds_curvatures se llena cada row con la lista speed_curvatures.
# Para df_runs_number_curve se llena cada row con una lista run_curve.


# Para df_peaks_avg_curve se llena cada row usando la lista avg_max que contiene 
# los promedios MAX por cada corrida.

# Para df_abs_avg_curve se llena cada row usando la lista abs_avg_curve que contiene
# los promedios ABS por cada corrida. 


for i in range(3):
    df_perf_type.loc[i,'Online Performance'] = 'Curve'
    df_speeds_curvatures.loc[i,'Speed / Curvature'] = speeds_curvatures[i]
    df_runs_number_curve.loc[i,'Run #'] = runs_curve[i]
    df_peaks_avg_curve.loc[i,'Max Average XTE  of runs (m)'] = avg_max[i]
    df_abs_avg_curve.loc[i, 'Absolute Average XTE of runs (m)'] = avg_abs_results[i]
    df_std_avg_curve.loc[i, 'Standard Deviation Average of runs (m)'] = avg_std_results[i]

In [277]:
df_output_curve = pd.concat([
                              df_perf_type,
                              df_speeds_curvatures,
                              df_runs_number_curve,
                              df_peaks_avg_curve,
                              df_abs_avg_curve,
                              df_std_avg_curve], axis=1)

In [278]:
df_output_curve

,Online Performance,Speed / Curvature,Run #,Max Average XTE of runs (m),Absolute Average XTE of runs (m),Standard Deviation Average of runs (m)
0,Curve,5mph / 1 deg/m,1,0.114,0.036,0.044
1,Curve,5 mph / 2deg/m,2,0.229,0.07,0.088
2,Curve,8mph / 1 deg/m,3,0.153,0.052,0.062


In [279]:
# Se exportan los dataframes al mismo excel file. 
# Sheet name nos sirve para especificar el nombre que llevara la sheet donde se exportara el df


rows_spacing = 2

with pd.ExcelWriter(filename_excel_report) as writer:
   
    df_online.to_excel(writer, sheet_name= 'Online Runs',index=False)
    
    df_output_online.to_excel(writer, sheet_name= 'Online Runs',
                              startrow= len(df_online) + rows_spacing + 1,index=False)


    df_curve.to_excel(writer, sheet_name= 'Curve Runs',index=False)
    df_output_curve.to_excel(writer, sheet_name= 'Curve Runs',
                             startrow= len(df_curve) + rows_spacing + 1, index=False)
    


In [280]:
df_output_online

,Online Performance,Speed,Run #,Max Average XTE of runs (m),Absolute Average XTE of runs (m),Standard Deviation Average of runs (m)
0,Straight,1.25mph,1,0.034,0.012,0.014
1,Straight,3.1mph,2,0.04,0.015,0.018
2,Straight,6.2mph,3,0.109,0.043,0.053
3,Straight,9.3mph,4,0.205,0.075,0.089
4,Straight,12.5mph,5,0.218,0.088,0.102


In [281]:
df_output_curve

,Online Performance,Speed / Curvature,Run #,Max Average XTE of runs (m),Absolute Average XTE of runs (m),Standard Deviation Average of runs (m)
0,Curve,5mph / 1 deg/m,1,0.114,0.036,0.044
1,Curve,5 mph / 2deg/m,2,0.229,0.07,0.088
2,Curve,8mph / 1 deg/m,3,0.153,0.052,0.062


Si genera este error, revisa que los rows donde realizara la comparacion sean los correctos


TypeError: '>' not supported between instances of 'NoneType' and 'float'

In [282]:
#Se abre el archivo usando openpyxl

#Aqui solo se crea la instancia para manipular el excel usando la libreria openpyxl
wb_file = load_workbook(filename_excel_report) 




#Se crean los colores para cada celda
red_code = "FFC7CE"


# start color: Excel pinta la celda usando cierto patron, ya sea puntos o lineas,
# start color es en palabras resumidas el color de esos puntos con los que se llena la celda



red_fill = PatternFill(start_color = red_code,
                       fill_type = 'solid' )



#Se selecciona la hora con las corridas online - Online Runs SHEET
 
# Parameters

online_runs_sheet = wb_file['Online Runs']  # Hoja que contiene las corridas online en el excel
row_start_online = 19                               # First row 
row_end_online = 24                                 # Last row (NO - inclusive)


for idx_row in range(row_start_online,row_end_online):
        
        cell_peaks = online_runs_sheet.cell(row = idx_row, column = 4) # column 4 == es la columna D
        cell_abs = online_runs_sheet.cell(row = idx_row, column = 5) # column 5 == es la columna E
        cell_std = online_runs_sheet.cell(row = idx_row, column = 6) # column 6 == es la columna E


        if cell_peaks.value > spec_peaks:
            cell_peaks.fill = red_fill
        
        if cell_abs.value > spec_abs:
              cell_abs.fill = red_fill

        if cell_std.value > spec_std:
              cell_std.fill = red_fill
              

        

#Se selecciona la hora con las corridas  - Curve Runs SHEET

# Parameters

curve_runs_sheet = wb_file['Curve Runs']    # Hoja que contiene las corridas curve en el excel
row_start_curve = 14                               # First row 
row_end_curve = 17                                 # Last row (NO - inclusive)



for idx_row in range(row_start_curve,row_end_curve):
        
        cell_peaks = curve_runs_sheet.cell(row = idx_row, column = 4) # column 4 == es la columna D
        cell_abs = curve_runs_sheet.cell(row = idx_row, column = 5) # column 5 == es la columna E
        cell_std = curve_runs_sheet.cell(row = idx_row, column = 6) # column 6 == es la columna E


        if cell_peaks.value > spec_peaks:
            cell_peaks.fill = red_fill
        
        if cell_abs.value > spec_abs:
              cell_abs.fill = red_fill

        if cell_std.value > spec_std:
              cell_std.fill = red_fill
        



wb_file.save(filename_excel_report)